# Construyendo tu Red Neuronal Profunda: Paso a Paso

Bienvenido a la tarea de la semana 4 (parte 1 de 2). Anteriormente entrenaste una Red Neuronal de 2 capas con una sola capa oculta. Esta semana, construiras una red neuronal profunda con tantas capas como quieras!

- En este notebook, implementaras todas las funciones necesarias para construir una red neuronal profunda.
- En la siguiente tarea, usaras estas funciones para construir una red neuronal profunda para clasificacion de imagenes.

**Al final de esta tarea, seras capaz de:**

- Usar unidades no lineales como ReLU para mejorar tu modelo
- Construir una red neuronal mas profunda (con mas de 1 capa oculta)
- Implementar una clase de red neuronal facil de usar

**Notacion**:
- El superindice $[l]$ denota una cantidad asociada con la capa $l^{th}$. 
    - Ejemplo: $a^{[L]}$ es la activacion de la capa $L^{th}$. $W^{[L]}$ y $b^{[L]}$ son los parametros de la capa $L^{th}$.
- El superindice $(i)$ denota una cantidad asociada con el ejemplo $i^{th}$. 
    - Ejemplo: $x^{(i)}$ es el ejemplo de entrenamiento $i^{th}$.
- El subindice $i$ denota la entrada $i^{th}$ de un vector.
    - Ejemplo: $a^{[l]}_i$ denota la entrada $i^{th}$ de las activaciones de la capa $l^{th}$.

Comencemos!

## Nota Importante sobre el Envio al AutoGrader

Antes de enviar tu tarea al AutoGrader, asegurate de no estar haciendo lo siguiente:

1. No has agregado ninguna instruccion `print` _extra_ en la tarea.
2. No has agregado ninguna celda de codigo _extra_ en la tarea.
3. No has cambiado ninguno de los parametros de las funciones.
4. No estas usando variables globales dentro de tus ejercicios calificados. A menos que se indique especificamente, por favor abstente de hacerlo y usa variables locales en su lugar.
5. No estas cambiando el codigo de la tarea donde no es necesario, como crear variables _extra_.

Si haces alguna de las anteriores, obtendras algo como `Grader Error: Grader feedback not found` (o un error inesperado similar) al enviar tu tarea. Antes de pedir ayuda/depurar los errores en tu tarea, verifica esto primero. Si este es el caso, y no recuerdas los cambios que has hecho, puedes obtener una copia nueva de la tarea siguiendo estas [instrucciones](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/iLwon/h-ow-to-refresh-your-workspace).

## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Esquema General](#2)
- [3 - Inicializacion](#3)
    - [3.1 - Red Neuronal de 2 capas](#3-1)
        - [Ejercicio 1 - initialize_parameters](#ex-1)
    - [3.2 - Red Neuronal de L capas](#3-2)
        - [Ejercicio 2 - initialize_parameters_deep](#ex-2)
- [4 - Modulo de Propagacion Hacia Adelante](#4)
    - [4.1 - Parte Lineal](#4-1)
        - [Ejercicio 3 - linear_forward](#ex-3)
    - [4.2 - Activacion Lineal](#4-2)
        - [Ejercicio 4 - linear_activation_forward](#ex-4)
    - [4.3 - Modelo de L Capas](#4-3)
        - [Ejercicio 5 - L_model_forward](#ex-5)
- [5 - Funcion de Costo](#5)
    - [Ejercicio 6 - compute_cost](#ex-6)
- [6 - Modulo de Retropropagacion](#6)
    - [6.1 - Retropropagacion Lineal](#6-1)
        - [Ejercicio 7 - linear_backward](#ex-7)
    - [6.2 - Retropropagacion Lineal-Activacion](#6-2)
        - [Ejercicio 8 - linear_activation_backward](#ex-8)
    - [6.3 - Retropropagacion del Modelo L](#6-3)
        - [Ejercicio 9 - L_model_backward](#ex-9)
    - [6.4 - Actualizar Parametros](#6-4)
        - [Ejercicio 10 - update_parameters](#ex-10)

<a name='1'></a>
## 1 - Paquetes

Primero, importa todos los paquetes que necesitaras durante esta tarea. 

- [numpy](www.numpy.org) es el paquete principal para computacion cientifica con Python.
- [matplotlib](http://matplotlib.org) es una biblioteca para graficar en Python.
- dnn_utils proporciona algunas funciones necesarias para este notebook.
- testCases proporciona algunos casos de prueba para evaluar la correctitud de tus funciones.
- np.random.seed(1) se usa para mantener todas las llamadas a funciones aleatorias consistentes. Ayuda a calificar tu trabajo. Por favor no cambies la semilla! 

In [1]:
### v1.2

In [3]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<a name='2'></a>
## 2 - Esquema General

Para construir tu red neuronal, implementaras varias "funciones auxiliares". Estas funciones auxiliares se usaran en la siguiente tarea para construir una red neuronal de dos capas y una red neuronal de L capas. 

Cada pequena funcion auxiliar tendra instrucciones detalladas para guiarte a traves de los pasos necesarios. Aqui hay un esquema de los pasos en esta tarea:

- Inicializar los parametros para una red de dos capas y para una red neuronal de $L$ capas
- Implementar el modulo de propagacion hacia adelante (mostrado en morado en la figura de abajo)
     - Completar la parte LINEAL del paso de propagacion hacia adelante de una capa (resultando en $Z^{[l]}$).
     - La funcion de ACTIVACION se te proporciona (relu/sigmoid)
     - Combinar los dos pasos anteriores en una nueva funcion [LINEAL->ACTIVACION].
     - Apilar la funcion [LINEAL->RELU] L-1 veces (para las capas 1 hasta L-1) y agregar un [LINEAL->SIGMOID] al final (para la capa final $L$). Esto te da una nueva funcion L_model_forward.
- Calcular la perdida
- Implementar el modulo de retropropagacion (mostrado en rojo en la figura de abajo)
    - Completar la parte LINEAL del paso de retropropagacion de una capa
    - Se te proporciona el gradiente de la funcion de ACTIVACION (relu_backward/sigmoid_backward) 
    - Combinar los dos pasos anteriores en una nueva funcion [LINEAL->ACTIVACION] hacia atras
    - Apilar [LINEAL->RELU] hacia atras L-1 veces y agregar [LINEAL->SIGMOID] hacia atras en una nueva funcion L_model_backward
- Finalmente, actualizar los parametros

<img src="images/final outline.png" style="width:800px;height:500px;">
<caption><center><b>Figura 1</b></center></caption><br>


**Nota**:

Para cada funcion hacia adelante, hay una funcion hacia atras correspondiente. Por eso en cada paso de tu modulo hacia adelante almacenaras algunos valores en un cache. Estos valores en cache son utiles para calcular gradientes. 

En el modulo de retropropagacion, puedes usar el cache para calcular los gradientes. No te preocupes, esta tarea te mostrara exactamente como llevar a cabo cada uno de estos pasos! 

<a name='3'></a>
## 3 - Inicializacion

Escribiras dos funciones auxiliares para inicializar los parametros de tu modelo. La primera funcion se usara para inicializar parametros para un modelo de dos capas. La segunda generaliza este proceso de inicializacion a $L$ capas.

<a name='3-1'></a>
### 3.1 - Red Neuronal de 2 capas

<a name='ex-1'></a>
### Ejercicio 1 - initialize_parameters

Crea e inicializa los parametros de la red neuronal de 2 capas.

**Instrucciones**:

- La estructura del modelo es: *LINEAL -> RELU -> LINEAL -> SIGMOID*. 
- Usa esta inicializacion aleatoria para las matrices de pesos: `np.random.randn(d0, d1, ..., dn) * 0.01` con la forma correcta. La documentacion de [np.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html)
- Usa inicializacion en cero para los sesgos: `np.zeros(shape)`. La documentacion de [np.zeros](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html)

In [4]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    Argument:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer
    
    Returns:
    parameters -- python dictionary containing your parameters:
                    W1 -- weight matrix of shape (n_h, n_x)
                    b1 -- bias vector of shape (n_h, 1)
                    W2 -- weight matrix of shape (n_y, n_h)
                    b2 -- bias vector of shape (n_y, 1)
    """
    
    np.random.seed(1)
    
    #Explicación rápida
    
    # W1 y W2: Se inicializan con valores aleatorios muy pequeños (* 0.01) para romper la simetría. Si todos los pesos fueran iguales, todas las neuronas aprenderían lo mismo.
    # b1 y b2: Se inicializan en cero, lo cual está bien porque la simetría ya se rompe con los pesos W.
    # Las shapes siguen la regla: W[l] tiene forma (capa_actual, capa_anterior) y b[l] tiene forma (capa_actual, 1).
    
    #(≈ 4 lines of code)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    # YOUR CODE ENDS HERE
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

In [5]:
print("Test Case 1:\n")
parameters = initialize_parameters(3,2,1)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_1(initialize_parameters)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters(4,3,2)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_2(initialize_parameters)

Test Case 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.


***Salida esperada***
```
Test Case 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - Red Neuronal de L capas

La inicializacion para una red neuronal mas profunda de L capas es mas complicada porque hay muchas mas matrices de pesos y vectores de sesgo. Al completar la funcion `initialize_parameters_deep`, debes asegurarte de que tus dimensiones coincidan entre cada capa. Recuerda que $n^{[l]}$ es el numero de unidades en la capa $l$. Por ejemplo, si el tamano de tu entrada $X$ es $(12288, 209)$ (con $m=209$ ejemplos) entonces:

<table style="width:100%">
    <tr>
        <td>  </td> 
        <td> <b>Forma de W</b> </td> 
        <td> <b>Forma de b</b>  </td> 
        <td> <b>Activacion</b> </td>
        <td> <b>Forma de la Activacion</b> </td> 
    <tr>
    <tr>
        <td> <b>Capa 1</b> </td> 
        <td> $(n^{[1]},12288)$ </td> 
        <td> $(n^{[1]},1)$ </td> 
        <td> $Z^{[1]} = W^{[1]}  X + b^{[1]} $ </td> 
        <td> $(n^{[1]},209)$ </td> 
    <tr>
    <tr>
        <td> <b>Capa 2</b> </td> 
        <td> $(n^{[2]}, n^{[1]})$  </td> 
        <td> $(n^{[2]},1)$ </td> 
        <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td> 
        <td> $(n^{[2]}, 209)$ </td> 
    <tr>
       <tr>
        <td> $\vdots$ </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$</td> 
        <td> $\vdots$  </td> 
    <tr>  
   <tr>
       <td> <b>Capa L-1</b> </td> 
        <td> $(n^{[L-1]}, n^{[L-2]})$ </td> 
        <td> $(n^{[L-1]}, 1)$  </td> 
        <td>$Z^{[L-1]} =  W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td> 
        <td> $(n^{[L-1]}, 209)$ </td> 
   <tr>
   <tr>
       <td> <b>Capa L</b> </td> 
        <td> $(n^{[L]}, n^{[L-1]})$ </td> 
        <td> $(n^{[L]}, 1)$ </td>
        <td> $Z^{[L]} =  W^{[L]} A^{[L-1]} + b^{[L]}$</td>
        <td> $(n^{[L]}, 209)$  </td> 
    <tr>
</table>



Recuerda que cuando calculas $W X + b$ en Python, se realiza broadcasting. Por ejemplo, si: 

$$ W = \begin{bmatrix}
    w_{00}  & w_{01} & w_{02} \\
    w_{10}  & w_{11} & w_{12} \\
    w_{20}  & w_{21} & w_{22} 
\end{bmatrix}\;\;\; X = \begin{bmatrix}
    x_{00}  & x_{01} & x_{02} \\
    x_{10}  & x_{11} & x_{12} \\
    x_{20}  & x_{21} & x_{22} 
\end{bmatrix} \;\;\; b =\begin{bmatrix}
    b_0  \\
    b_1  \\
    b_2
\end{bmatrix}\tag{2}$$

Entonces $WX + b$ sera:

$$ WX + b = \begin{bmatrix}
    (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
    (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
    (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 &  (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3}  $$

<a name='ex-2'></a>
### Ejercicio 2 - initialize_parameters_deep

Implementa la inicializacion para una Red Neuronal de L capas. 

**Instrucciones**:
- La estructura del modelo es *[LINEAL -> RELU] $ \times$ (L-1) -> LINEAL -> SIGMOID*. Es decir, tiene $L-1$ capas usando una funcion de activacion ReLU seguida de una capa de salida con una funcion de activacion sigmoid.
- Usa inicializacion aleatoria para las matrices de pesos. Usa `np.random.randn(d0, d1, ..., dn) * 0.01`.
- Usa inicializacion en ceros para los sesgos. Usa `np.zeros(shape)`.
- Almacenaras $n^{[l]}$, el numero de unidades en las diferentes capas, en una variable `layer_dims`. Por ejemplo, los `layer_dims` para el modelo de clasificacion de datos planares de la semana pasada habrian sido [2,4,1]: Habia dos entradas, una capa oculta con 4 unidades ocultas, y una capa de salida con 1 unidad de salida. Esto significa que la forma de `W1` era (4,2), `b1` era (4,1), `W2` era (1,4) y `b2` era (1,1). Ahora generalizaras esto a $L$ capas! 
- Aqui esta la implementacion para $L=1$ (red neuronal de una capa). Deberia inspirarte para implementar el caso general (red neuronal de L capas).
```python
    if L == 1:
        parameters["W" + str(L)] = np.random.randn(layer_dims[1], layer_dims[0]) * 0.01
        parameters["b" + str(L)] = np.zeros((layer_dims[1], 1))
```

In [ ]:
# GRADED FUNCTION: initialize_parameters_deep

def initialize_parameters_deep(layer_dims):
    """
    Arguments:
    layer_dims -- python array (list) containing the dimensions of each layer in our network
    
    Returns:
    parameters -- python dictionary containing your parameters "W1", "b1", ..., "WL", "bL":
                    Wl -- weight matrix of shape (layer_dims[l], layer_dims[l-1])
                    bl -- bias vector of shape (layer_dims[l], 1)
    """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # number of layers in the network

    for l in range(1, L):
        
        #Explicación
        #Es exactamente lo mismo que el ejercicio 1, pero dentro del bucle for l in range(1, L):

        #l recorre desde 1 hasta L-1 (cada capa)
        #layer_dims[l] = número de neuronas en la capa actual
        #layer_dims[l-1] = número de neuronas en la capa anterior
        #Se guardan en el diccionario con claves "W1", "b1", "W2", "b2", etc., usando str(l)
        
        #(≈ 2 lines of code)
        # parameters['W' + str(l)] = ...
        # parameters['b' + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

        
        # YOUR CODE ENDS HERE
        
        assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

In [7]:
print("Test Case 1:\n")
parameters = initialize_parameters_deep([5,4,3])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_deep_test_1(initialize_parameters_deep)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters_deep([4,3,2])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

Test Case 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.


***Salida esperada***
```
Test Case 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - Modulo de Propagacion Hacia Adelante

<a name='4-1'></a>
### 4.1 - Parte Lineal 

Ahora que has inicializado tus parametros, puedes hacer el modulo de propagacion hacia adelante. Comenzaras implementando algunas funciones basicas que puedes usar de nuevo mas adelante al implementar el modelo. Ahora, completaras tres funciones en este orden:

- LINEAL
- LINEAL -> ACTIVACION donde ACTIVACION sera ReLU o Sigmoid. 
- [LINEAL -> RELU] $\times$ (L-1) -> LINEAL -> SIGMOID (modelo completo)

El modulo lineal hacia adelante (vectorizado sobre todos los ejemplos) calcula las siguientes ecuaciones:

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{4}$$

donde $A^{[0]} = X$. 

<a name='ex-3'></a>
### Ejercicio 3 - linear_forward 

Construye la parte lineal de la propagacion hacia adelante.

**Recordatorio**:
La representacion matematica de esta unidad es $Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}$. Tambien puede serte util `np.dot()`. Si tus dimensiones no coinciden, imprimir `W.shape` puede ayudar.

In [8]:
# GRADED FUNCTION: linear_forward

# Explicación
# Implementa la fórmula de la parte lineal de la propagación hacia adelante:

# $$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$$

# np.dot(W, A) = multiplicación de matrices $W \cdot A$
# + b = se suma el sesgo (numpy aplica broadcasting automáticamente)
# El resultado Z es lo que luego se pasa a la función de activación (ReLU o sigmoid)

def linear_forward(A, W, b):
    """
    Implement the linear part of a layer's forward propagation.

    Arguments:
    A -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)

    Returns:
    Z -- the input of the activation function, also called pre-activation parameter 
    cache -- a python tuple containing "A", "W" and "b" ; stored for computing the backward pass efficiently
    """
    
    #(≈ 1 line of code)
    # Z = ...
    # YOUR CODE STARTS HERE
    
    Z = np.dot(W, A) + b
    
    # YOUR CODE ENDS HERE
    cache = (A, W, b)
    
    return Z, cache

In [9]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))

linear_forward_test(linear_forward)

Z = [[ 3.26295337 -1.23429987]]
 All tests passed.


***Salida esperada***
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - Activacion Lineal 

En este notebook, usaras dos funciones de activacion:

- **Sigmoid**: $\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$. Se te ha proporcionado la funcion `sigmoid` que retorna **dos** elementos: el valor de activacion "`a`" y un "`cache`" que contiene "`Z`" (es lo que alimentaremos a la funcion hacia atras correspondiente). Para usarla simplemente llama: 
``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU**: La formula matematica para ReLU es $A = RELU(Z) = max(0, Z)$. Se te ha proporcionado la funcion `relu`. Esta funcion retorna **dos** elementos: el valor de activacion "`A`" y un "`cache`" que contiene "`Z`" (es lo que alimentaras a la funcion hacia atras correspondiente). Para usarla simplemente llama:
``` python
A, activation_cache = relu(Z)
```

Para mayor comodidad, vas a agrupar dos funciones (Lineal y Activacion) en una sola funcion (LINEAL->ACTIVACION). Por lo tanto, implementaras una funcion que realice el paso LINEAL hacia adelante, seguido de un paso de ACTIVACION hacia adelante.

<a name='ex-4'></a>
### Ejercicio 4 - linear_activation_forward

Implementa la propagacion hacia adelante de la capa *LINEAL->ACTIVACION*. La relacion matematica es: $A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} +b^{[l]})$ donde la activacion "g" puede ser sigmoid() o relu(). Usa `linear_forward()` y la funcion de activacion correcta.

In [ ]:
# GRADED FUNCTION: linear_activation_forward

# Explicación
# Combina dos pasos en una sola función:

# Parte lineal: Z = W·A_prev + b (usando linear_forward del ejercicio 3)
# Activación: aplica sigmoid(Z) o relu(Z) según el parámetro activation
# Ambos casos son idénticos excepto por la función de activación. Se guardan dos caches:

# linear_cache = (A_prev, W, b) — para el backpropagation lineal
# activation_cache = Z — para el backpropagation de la activación


def linear_activation_forward(A_prev, W, b, activation):
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer

    Arguments:
    A_prev -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"

    Returns:
    A -- the output of the activation function, also called the post-activation value 
    cache -- a python tuple containing "linear_cache" and "activation_cache";
             stored for computing the backward pass efficiently
    """
    
    if activation == "sigmoid":
        #(≈ 2 lines of code)
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        
        Z, linear_cache = linear_forward(A_prev, W, b)
        A, activation_cache = sigmoid(Z)
        
        # YOUR CODE ENDS HERE
    
    elif activation == "relu":
        #(≈ 2 lines of code)
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        
        Z, linear_cache = linear_forward(A_prev, W, b)
        A, activation_cache = relu(Z)
        
        # YOUR CODE ENDS HERE
    cache = (linear_cache, activation_cache)

    return A, cache

In [11]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("With sigmoid: A = " + str(t_A))

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("With ReLU: A = " + str(t_A))

linear_activation_forward_test(linear_activation_forward)

With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
 All tests passed.


***Salida esperada***
```
With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
```

**Nota**: En deep learning, la computacion "[LINEAL->ACTIVACION]" se cuenta como una sola capa en la red neuronal, no dos capas. 

<a name='4-3'></a>
### 4.3 - Modelo de L Capas 

Para aun *mas* comodidad al implementar la Red Neuronal de $L$ capas, necesitaras una funcion que replique la anterior (`linear_activation_forward` con RELU) $L-1$ veces, y luego la siga con un `linear_activation_forward` con SIGMOID.

<img src="images/model_architecture_kiank.png" style="width:600px;height:300px;">
<caption><center> <b>Figura 2</b> : Modelo *[LINEAL -> RELU] $\times$ (L-1) -> LINEAL -> SIGMOID*</center></caption><br>

<a name='ex-5'></a>
### Ejercicio 5 - L_model_forward

Implementa la propagacion hacia adelante del modelo anterior.

**Instrucciones**: En el codigo de abajo, la variable `AL` denotara $A^{[L]} = \sigma(Z^{[L]}) = \sigma(W^{[L]} A^{[L-1]} + b^{[L]})$. (Esto a veces tambien se llama `Yhat`, es decir, esto es $\hat{Y}$.) 

**Pistas**:
- Usa las funciones que escribiste previamente 
- Usa un bucle for para replicar [LINEAL->RELU] (L-1) veces
- No olvides llevar un registro de los caches en la lista "caches". Para agregar un nuevo valor `c` a una `lista`, puedes usar `list.append(c)`.

In [14]:
# GRADED FUNCTION: L_model_forward

# Explicación
# mplementa la arquitectura completa: [LINEAR → RELU] × (L-1) → LINEAR → SIGMOID

# El bucle (l = 1 hasta L-1): recorre las capas ocultas, cada una con activación ReLU
# Después del bucle: la última capa (L) usa sigmoid porque es un problema de clasificación binaria (la salida debe estar entre 0 y 1)
# En cada paso se guarda el cache en la lista caches para usarlo luego en el backpropagation

def L_model_forward(X, parameters):
    """
    Implement forward propagation for the [LINEAR->RELU]*(L-1)->LINEAR->SIGMOID computation
    
    Arguments:
    X -- data, numpy array of shape (input size, number of examples)
    parameters -- output of initialize_parameters_deep()
    
    Returns:
    AL -- activation value from the output (last) layer
    caches -- list of caches containing:
                every cache of linear_activation_forward() (there are L of them, indexed from 0 to L-1)
    """

    caches = []
    A = X
    L = len(parameters) // 2                  # number of layers in the neural network
    
    # Implement [LINEAR -> RELU]*(L-1). Add "cache" to the "caches" list.
    # The for loop starts at 1 because layer 0 is the input
    for l in range(1, L):
        A_prev = A 
        #(≈ 2 lines of code)
        # A, cache = ...
        # caches ...
        # YOUR CODE STARTS HERE
        
        A, cache = linear_activation_forward(A_prev, parameters['W' + str(l)], parameters['b' + str(l)], activation="relu")
        caches.append(cache)    
        
        # YOUR CODE ENDS HERE
    
    # Implement LINEAR -> SIGMOID. Add "cache" to the "caches" list.
    #(≈ 2 lines of code)
    # AL, cache = ...
    # caches ...
    # YOUR CODE STARTS HERE
    
    AL, cache = linear_activation_forward(A, parameters['W' + str(L)], parameters['b' + str(L)], activation="sigmoid")
    caches.append(cache)
    
    # YOUR CODE ENDS HERE
          
    return AL, caches

In [15]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)

print("AL = " + str(t_AL))

L_model_forward_test(L_model_forward)

AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
 All tests passed.


***Salida esperada***
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

**Excelente!** Has implementado una propagacion hacia adelante completa que toma la entrada X y produce un vector fila $A^{[L]}$ que contiene tus predicciones. Tambien registra todos los valores intermedios en "caches". Usando $A^{[L]}$, puedes calcular el costo de tus predicciones.

<a name='5'></a>
## 5 - Funcion de Costo

Ahora puedes implementar la propagacion hacia adelante y hacia atras! Necesitas calcular el costo, para verificar si tu modelo realmente esta aprendiendo.

<a name='ex-6'></a>
### Ejercicio 6 - compute_cost
Calcula el costo de entropia cruzada $J$, usando la siguiente formula: 

$$-\frac{1}{m} \sum\limits_{i = 1}^{m} (y^{(i)}\log\left(a^{[L] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right)) \tag{7}$$


In [16]:
# GRADED FUNCTION: compute_cost

def compute_cost(AL, Y):
    """
    Implement the cost function defined by equation (7).

    Arguments:
    AL -- probability vector corresponding to your label predictions, shape (1, number of examples)
    Y -- true "label" vector (for example: containing 0 if non-cat, 1 if cat), shape (1, number of examples)

    Returns:
    cost -- cross-entropy cost
    """
    
    m = Y.shape[1]

    # Compute loss from aL and y.
    # (≈ 1 lines of code)
    # cost = ...
    # YOUR CODE STARTS HERE
    
    cost = -1/m * np.sum(Y * np.log(AL) + (1 - Y) * np.log(1 - AL))
    
    # YOUR CODE ENDS HERE
    
    cost = np.squeeze(cost)      # To make sure your cost's shape is what we expect (e.g. this turns [[17]] into 17).

    
    return cost

In [17]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)

print("Cost: " + str(t_cost))

compute_cost_test(compute_cost)

Cost: 0.2797765635793422
 All tests passed.


**Salida Esperada**:

<table>
    <tr>
        <td><b>costo</b> </td>
    <td> 0.2797765635793422</td> 
    </tr>
</table>

<a name='6'></a>
## 6 - Modulo de Retropropagacion

Al igual que hiciste para la propagacion hacia adelante, implementaras funciones auxiliares para la retropropagacion. Recuerda que la retropropagacion se usa para calcular el gradiente de la funcion de perdida con respecto a los parametros. 

**Recordatorio**: 
<img src="images/backprop_kiank.png" style="width:650px;height:250px;">
<caption><center><font color='purple'><b>Figura 3</b>: Propagacion hacia adelante y hacia atras para LINEAL->RELU->LINEAL->SIGMOID <br> <i>Los bloques morados representan la propagacion hacia adelante, y los bloques rojos representan la retropropagacion.</font></center></caption>


Ahora, de manera similar a la propagacion hacia adelante, vas a construir la retropropagacion en tres pasos:
1. LINEAL hacia atras
2. LINEAL -> ACTIVACION hacia atras donde ACTIVACION calcula la derivada de la activacion ReLU o sigmoid
3. [LINEAL -> RELU] $\times$ (L-1) -> LINEAL -> SIGMOID hacia atras (modelo completo)

Para el siguiente ejercicio, necesitaras recordar que:

- `b` es una matriz (np.ndarray) con 1 columna y n filas, es decir: b = [[1.0], [2.0]] (recuerda que `b` es una constante)
- np.sum realiza una suma sobre los elementos de un ndarray
- axis=1 o axis=0 especifican si la suma se realiza por filas o por columnas respectivamente
- keepdims especifica si se deben mantener las dimensiones originales de la matriz.
- Mira el siguiente ejemplo para aclarar:

In [18]:
A = np.array([[1, 2], [3, 4]])

print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))
print('axis=0 and keepdims=True')
print(np.sum(A, axis=0, keepdims=True))
print('axis=0 and keepdims=False')
print(np.sum(A, axis=0, keepdims=False))

axis=1 and keepdims=True
[[3]
 [7]]
axis=1 and keepdims=False
[3 7]
axis=0 and keepdims=True
[[4 6]]
axis=0 and keepdims=False
[4 6]


<a name='6-1'></a>
### 6.1 - Retropropagacion Lineal

Para la capa $l$, la parte lineal es: $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ (seguida de una activacion).

Supongamos que ya has calculado la derivada $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$. Quieres obtener $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$.

<img src="images/linearback_kiank.png" style="width:250px;height:300px;">
<caption><center><font color='purple'><b>Figura 4</b></font></center></caption>

Las tres salidas $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ se calculan usando la entrada $dZ^{[l]}$.

Aqui estan las formulas que necesitas:
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{8}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{9}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]} \tag{10}$$


$A^{[l-1] T}$ es la transpuesta de $A^{[l-1]}$. 

<a name='ex-7'></a>
### Ejercicio 7 - linear_backward 

Usa las 3 formulas anteriores para implementar `linear_backward()`.

**Pista**:

- En numpy puedes obtener la transpuesta de un ndarray `A` usando `A.T` o `A.transpose()`

In [22]:
# GRADED FUNCTION: linear_backward

# Explicación
# Implementa las 3 fórmulas del backpropagation lineal:

# Gradiente	Fórmula	Código
# $dW^{[l]}$	$\frac{1}{m} dZ^{[l]} \cdot A^{[l-1]T}$	np.dot(dZ, A_prev.T) / m
# $db^{[l]}$	$\frac{1}{m} \sum dZ^{[l]}$	np.sum(dZ, axis=1, keepdims=True) / m
# $dA^{[l-1]}$	$W^{[l]T} \cdot dZ^{[l]}$	np.dot(W.T, dZ)
# A_prev.T → transpone para que las dimensiones coincidan en la multiplicación
# axis=1, keepdims=True → suma por filas y mantiene la forma (n, 1) para db
# W.T → transpone W para propagar el gradiente hacia la capa anterior


def linear_backward(dZ, cache):
    """
    Implement the linear portion of backward propagation for a single layer (layer l)

    Arguments:
    dZ -- Gradient of the cost with respect to the linear output (of current layer l)
    cache -- tuple of values (A_prev, W, b) coming from the forward propagation in the current layer

    Returns:
    dA_prev -- Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev
    dW -- Gradient of the cost with respect to W (current layer l), same shape as W
    db -- Gradient of the cost with respect to b (current layer l), same shape as b
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### START CODE HERE ### (≈ 3 lines of code)
    # dW = ...
    # db = ... sum by the rows of dZ with keepdims=True
    # dA_prev = ...
    # YOUR CODE STARTS HERE
    
    dW = 1/m * np.dot(dZ, A_prev.T)
    db = 1/m * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)

    
    # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [23]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)

print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))

linear_backward_test(linear_backward)

dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 All tests passed.


**Salida Esperada**:
```
dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 ```

<a name='6-2'></a>
### 6.2 - Retropropagacion Lineal-Activacion

A continuacion, crearas una funcion que combine las dos funciones auxiliares: **`linear_backward`** y el paso hacia atras de la activacion **`linear_activation_backward`**. 

Para ayudarte a implementar `linear_activation_backward`, se han proporcionado dos funciones hacia atras:
- **`sigmoid_backward`**: Implementa la retropropagacion para la unidad SIGMOID. Puedes llamarla asi:

```python
dZ = sigmoid_backward(dA, activation_cache)
```

- **`relu_backward`**: Implementa la retropropagacion para la unidad RELU. Puedes llamarla asi:

```python
dZ = relu_backward(dA, activation_cache)
```

Si $g(.)$ es la funcion de activacion, 
`sigmoid_backward` y `relu_backward` calculan 

$$dZ^{[l]} = dA^{[l]} \ast g'(Z^{[l]}) \tag{11}$$ 

<a name='ex-8'></a>

### Ejercicio 8 - linear_activation_backward

Implementa la retropropagacion para la capa *LINEAL->ACTIVACION*.

In [ ]:
# GRADED FUNCTION: linear_activation_backward

# Explicación
# Es el inverso del ejercicio 4, ahora en dos pasos hacia atrás:

# Backprop de la activación: calcula $dZ$ usando relu_backward o sigmoid_backward
# Aplica: $dZ^{[l]} = dA^{[l]} * g'(Z^{[l]})$
# Backprop lineal: calcula dA_prev, dW, db usando linear_backward del ejercicio 7
# Igual que en el ejercicio 4, ambos casos son idénticos excepto por la función de activación.


def linear_activation_backward(dA, cache, activation):
    """
    Implement the backward propagation for the LINEAR->ACTIVATION layer.
    
    Arguments:
    dA -- post-activation gradient for current layer l 
    cache -- tuple of values (linear_cache, activation_cache) we store for computing backward propagation efficiently
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"
    
    Returns:
    dA_prev -- Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev
    dW -- Gradient of the cost with respect to W (current layer l), same shape as W
    db -- Gradient of the cost with respect to b (current layer l), same shape as b
    """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        #(≈ 2 lines of code)
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        
        dZ = relu_backward(dA, activation_cache)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)

        
        # YOUR CODE ENDS HERE
        
    elif activation == "sigmoid":
        #(≈ 2 lines of code)
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        
        dZ = sigmoid_backward(dA, activation_cache)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)

        
        # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [25]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("With sigmoid: dA_prev = " + str(t_dA_prev))
print("With sigmoid: dW = " + str(t_dW))
print("With sigmoid: db = " + str(t_db))

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("With relu: dA_prev = " + str(t_dA_prev))
print("With relu: dW = " + str(t_dW))
print("With relu: db = " + str(t_db))

linear_activation_backward_test(linear_activation_backward)

With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
 All tests passed.


**Salida esperada:**

```
With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - Retropropagacion del Modelo L 

Ahora implementaras la funcion hacia atras para toda la red! 

Recuerda que cuando implementaste la funcion `L_model_forward`, en cada iteracion almacenaste un cache que contiene (X, W, b y z). En el modulo de retropropagacion, usaras esas variables para calcular los gradientes. Por lo tanto, en la funcion `L_model_backward`, iteraras a traves de todas las capas ocultas hacia atras, comenzando desde la capa $L$. En cada paso, usaras los valores en cache de la capa $l$ para retropropagar a traves de la capa $l$. La Figura 5 de abajo muestra el paso hacia atras. 


<img src="images/mn_backward.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>Figura 5</b>: Paso hacia atras</font></center></caption>

**Inicializando la retropropagacion**:

Para retropropagar a traves de esta red, sabes que la salida es: 
$A^{[L]} = \sigma(Z^{[L]})$. Tu codigo necesita calcular `dAL` $= \frac{\partial \mathcal{L}}{\partial A^{[L]}}$.
Para hacerlo, usa esta formula (derivada usando calculo que, de nuevo, no necesitas conocimiento profundo de!):
```python
dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL)) # derivada del costo con respecto a AL
```

Luego puedes usar este gradiente post-activacion `dAL` para seguir hacia atras. Como se ve en la Figura 5, puedes alimentar `dAL` a la funcion LINEAL->SIGMOID hacia atras que implementaste (que usara los valores en cache almacenados por la funcion L_model_forward). 

Despues de eso, tendras que usar un bucle `for` para iterar a traves de todas las otras capas usando la funcion LINEAL->RELU hacia atras. Debes almacenar cada dA, dW y db en el diccionario grads. Para hacerlo, usa esta formula: 

$$grads["dW" + str(l)] = dW^{[l]}\tag{15} $$

Por ejemplo, para $l=3$ esto almacenaria $dW^{[l]}$ en `grads["dW3"]`.

<a name='ex-9'></a>
### Ejercicio 9 - L_model_backward

Implementa la retropropagacion para el modelo *[LINEAL->RELU] $\times$ (L-1) -> LINEAL -> SIGMOID*.

In [26]:
# GRADED FUNCTION: L_model_backward

def L_model_backward(AL, Y, caches):
    """
    Implement the backward propagation for the [LINEAR->RELU] * (L-1) -> LINEAR -> SIGMOID group
    
    Arguments:
    AL -- probability vector, output of the forward propagation (L_model_forward())
    Y -- true "label" vector (containing 0 if non-cat, 1 if cat)
    caches -- list of caches containing:
                every cache of linear_activation_forward() with "relu" (it's caches[l], for l in range(L-1) i.e l = 0...L-2)
                the cache of linear_activation_forward() with "sigmoid" (it's caches[L-1])
    
    Returns:
    grads -- A dictionary with the gradients
             grads["dA" + str(l)] = ... 
             grads["dW" + str(l)] = ...
             grads["db" + str(l)] = ... 
    """
    grads = {}
    L = len(caches) # the number of layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # after this line, Y is the same shape as AL
    
    # Initializing the backpropagation
    #(1 line of code)
    # dAL = ...
    # YOUR CODE STARTS HERE
    
    dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL))
    
    # YOUR CODE ENDS HERE
    
    # Lth layer (SIGMOID -> LINEAR) gradients. Inputs: "dAL, current_cache". Outputs: "grads["dAL-1"], grads["dWL"], grads["dbL"]
    #(approx. 5 lines)
    # current_cache = ...
    # dA_prev_temp, dW_temp, db_temp = ...
    # grads["dA" + str(L-1)] = ...
    # grads["dW" + str(L)] = ...
    # grads["db" + str(L)] = ...
    # YOUR CODE STARTS HERE
    
    current_cache = caches[L-1]
    dA_prev_temp, dW_temp, db_temp = linear_activation_backward(dAL, current_cache, activation="sigmoid")
    grads["dA" + str(L-1)] = dA_prev_temp
    grads["dW" + str(L)] = dW_temp
    grads["db" + str(L)] = db_temp

    
    # YOUR CODE ENDS HERE
    
    # Loop from l=L-2 to l=0
    for l in reversed(range(L-1)):
        # lth layer: (RELU -> LINEAR) gradients.
        # Inputs: "grads["dA" + str(l + 1)], current_cache". Outputs: "grads["dA" + str(l)] , grads["dW" + str(l + 1)] , grads["db" + str(l + 1)] 
        #(approx. 5 lines)
        # current_cache = ...
        # dA_prev_temp, dW_temp, db_temp = ...
        # grads["dA" + str(l)] = ...
        # grads["dW" + str(l + 1)] = ...
        # grads["db" + str(l + 1)] = ...
        # YOUR CODE STARTS HERE
        
        current_cache = caches[l]
        dA_prev_temp, dW_temp, db_temp = linear_activation_backward(grads["dA" + str(l + 1)], current_cache, activation="relu")
        grads["dA" + str(l)] = dA_prev_temp
        grads["dW" + str(l + 1)] = dW_temp
        grads["db" + str(l + 1)] = db_temp

        
        # YOUR CODE ENDS HERE

    return grads

In [27]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)

print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))

L_model_backward_test(L_model_backward)

dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
 All tests passed.


**Salida esperada:**

```
dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - Actualizar Parametros

En esta seccion, actualizaras los parametros del modelo usando descenso de gradiente: 

$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{16}$$
$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{17}$$

donde $\alpha$ es la tasa de aprendizaje. 

Despues de calcular los parametros actualizados, almacenalos en el diccionario parameters. 

<a name='ex-10'></a>
### Ejercicio 10 - update_parameters

Implementa `update_parameters()` para actualizar tus parametros usando descenso de gradiente.

**Instrucciones**:
Actualiza los parametros usando descenso de gradiente en cada $W^{[l]}$ y $b^{[l]}$ para $l = 1, 2, ..., L$. 

In [28]:
# GRADED FUNCTION: update_parameters

def update_parameters(params, grads, learning_rate):
    """
    Update parameters using gradient descent
    
    Arguments:
    params -- python dictionary containing your parameters 
    grads -- python dictionary containing your gradients, output of L_model_backward
    
    Returns:
    parameters -- python dictionary containing your updated parameters 
                  parameters["W" + str(l)] = ... 
                  parameters["b" + str(l)] = ...
    """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2 # number of layers in the neural network

    # Update rule for each parameter. Use a for loop.
    #(≈ 2 lines of code)
    for l in range(L):
        # parameters["W" + str(l+1)] = ...
        # parameters["b" + str(l+1)] = ...
        # YOUR CODE STARTS HERE
        
        parameters["W" + str(l+1)] = parameters["W" + str(l+1)] - learning_rate * grads["dW" + str(l+1)]
        parameters["b" + str(l+1)] = parameters["b" + str(l+1)] - learning_rate * grads["db" + str(l+1)]
        
        # YOUR CODE ENDS HERE
    return parameters

In [29]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)

print ("W1 = "+ str(t_parameters["W1"]))
print ("b1 = "+ str(t_parameters["b1"]))
print ("W2 = "+ str(t_parameters["W2"]))
print ("b2 = "+ str(t_parameters["b2"]))

update_parameters_test(update_parameters)

W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
 All tests passed.


**Salida esperada:**

```
W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
```

### Felicidades! 

Acabas de implementar todas las funciones necesarias para construir una red neuronal profunda, incluyendo: 

- Usar unidades no lineales para mejorar tu modelo
- Construir una red neuronal mas profunda (con mas de 1 capa oculta)
- Implementar una clase de red neuronal facil de usar

Esta fue de hecho una tarea larga, pero la siguiente parte de la tarea es mas facil. ;) 

En la siguiente tarea, pondras todo esto junto para construir dos modelos:

- Una red neuronal de dos capas
- Una red neuronal de L capas

De hecho, usaras estos modelos para clasificar imagenes de gato vs no-gato! (Miau!) Buen trabajo y nos vemos la proxima vez. 